# Objectif 

Recherche d'une situation passive favorable à la démonstration de la méthode de localisation dans le cas library actif / event passif.

Devant la difficulté rencontrée à trouver un passage de bateau AIS favorable avec la portion 5 on considère la seconde option consistant à utiliser les passages de l'ATALANTE. 

Date de création : 29/10/2026

In [ ]:
import os
import sys
import glob
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

from datetime import datetime

In [ ]:
if os.name == "nt":  # Windows
    ROOT_PROGRAM = r"C:\Users\baptiste.menetrier\Desktop\devPy\phd"
else:  # Linux
    ROOT_PROGRAM = "/home/program/ubf_tools"
sys.path.append(ROOT_PROGRAM)

from publication.publication_figure import set_subfigures_abc_labels, color
from real_data_analysis.acouplane.src.acouplane_cst import *
from real_data_analysis.acouplane.src.acouplane_utils import *


# Chargement des données 

## Données de la portion pour la librairie 
Les données sont stockées au format netcdf sous la forme d'un arbre (xarray DataTree) et regroupe l'ensemble des données associées à la portion d'intérêt. 

In [ ]:
portion_id = 5
fpath = f"{ACOUPLANE_PORTION_NC_ROOT_FPATH}_{portion_id}.nc"
portion = xr.open_datatree(fpath, auto_complex=True)
portion

## Données de positions 
Toutes les données disponibles : 

* Données AIS (traitées dans l'autre notebook)
* Données de l'ATALANTE 

In [ ]:
ds_pos = xr.open_dataset(ACOUPLANE_POSITIONS_NC_FPATH)
print(ds_pos)

# Étude des positions des sources passives 

## Définition d'une zone d'intérêt

On commence par définir la zone de recherche. Cette zone correspond à la zone du maillage de la librairie + un petit offset pour mieux visualiser la trajectoire des sources avant et après avoir traversé la grille. 

In [ ]:
# Step 1: select pos in the area of interest
lon_offset_percentage = lat_offset_percentage = 0.001

lon_min, lon_max = np.min([
    node.to_dataset().tir_src_lon.min() for _, node in portion.position.children.items()
]) * (1 - lon_offset_percentage / 2), np.max(
    [
    node.to_dataset().tir_src_lon.max() for _, node in portion.position.children.items()
]
) * (1 + lon_offset_percentage / 2)
lat_min, lat_max = np.min([
    node.to_dataset().tir_src_lat.min() for _, node in portion.position.children.items()
]) * (1 - lat_offset_percentage / 2), np.max(
    [
    node.to_dataset().tir_src_lat.max() for _, node in portion.position.children.items()
]
) * (1 + lat_offset_percentage / 2)
lon_min, lon_max, lat_min, lat_max

In [ ]:
# Illustrate on map
fig, ax = plot_portion_tir_pos(portion_tree=portion)
ax.add_patch(
    Rectangle(
        (lon_min, lat_min),
        lon_max - lon_min,
        lat_max - lat_min,
        fc="none",
        ec="k",
        lw=5,
        label="Selected area",
    )
)
ax.legend()
legend = ax.get_legend()

if legend is not None:
    handles = legend.legend_handles
    labels = [text.get_text() for text in legend.get_texts()]

    legend.remove()

    ax.legend(
        handles,
        labels,
        loc="upper right",
        fontsize=16,
        ncols=2,
    )

## Position ATL

On extrait ensuite les positions ATL qui passe dans la boite définie. 

In [ ]:
ds_atl = ds_pos.copy().drop_vars(
    [v for v in list(ds_pos.variables) if v not in ["atl_lon", "atl_lat", "atl_datetime"]]
)

mask = (
    (ds_atl["atl_lon"] >= lon_min)
    & (ds_atl["atl_lon"] <= lon_max)
    & (ds_atl["atl_lat"] >= lat_min)
    & (ds_atl["atl_lat"] <= lat_max)
)
ds_atl_in_area = ds_atl.where(mask)

# # Mask and drop coordinates that have no positions in the area
# ds_atl_in_area = (
#     ds_atl.where(mask).dropna("atl_datetime", how="all")
# )
# ds_atl_in_area

In [ ]:
# Plot
fig, ax = plot_portion_tir_pos(portion_tree=portion)
ax.add_patch(
    Rectangle(
        (lon_min, lat_min),
        lon_max - lon_min,
        lat_max - lat_min,
        fc="none",
        ec="k",
        lw=5,
        label="Selected area",
    )
)

ax.plot(ds_atl_in_area.atl_lon, ds_atl_in_area.atl_lat, "-x", markersize=2)

# for name, node in portion_event.position.children.items():
#     ds_position_section = node.to_dataset()
#     plt.plot(
#         ds_position_section.lon, ds_position_section.lat, "-x", markersize=2, label=f"Event {ds_position_section.event_id}"
#     )

ax.legend()
legend = ax.get_legend()

if legend is not None:
    handles = legend.legend_handles
    labels = [text.get_text() for text in legend.get_texts()]

    legend.remove()

    ax.legend(
        handles,
        labels,
        loc="upper right",
        fontsize=16,
        ncols=2,
    )

# Filtrage des trajectoires : disponibilité et viabilité des données acoustiques 

On filtre les trajectoires ATL en rejettant les trajectoires qui présentent un des deux critères suivant : 

* Les données acoustiques ne sont pas disponible (passage en dehors de la période d'acquisition)
* La trajectoires coincide avec une période d'activité de la source acoustique (canon)

## Création du DataTree avec toutes les trajectoires candidates 

Certain navire (MMSI = 61 par exemple) traversent plusieurs fois la zone. Pour la localisation il est préferable de distinguer chaque trajectoire. 

In [ ]:
# Cut into distinct events

nan_indicator = np.isnan(ds_atl_in_area.atl_lon * ds_atl_in_area.atl_lat)
edge_indicator = nan_indicator.astype(int)
rising_edge = np.diff(edge_indicator.values, append=edge_indicator.values[-1]) == 1
rising_edge_dt = ds_atl_in_area.atl_datetime.values[rising_edge]
falling_edge = np.diff(edge_indicator.values, append=edge_indicator.values[-1]) == -1
falling_edge_dt = ds_atl_in_area.atl_datetime.values[np.roll(falling_edge, +1)]       # Roll to avoid taking previosu point which is still a nan (diff = a[i+1] - a[i])

n_section_for_current_mmsi = max(rising_edge_dt.size, falling_edge_dt.size)

# if n_section_for_current_mmsi == 0:
#     continue 

# Edges might not be visible
if falling_edge_dt.size < n_section_for_current_mmsi:
    falling_edge_dt = np.concat([[ds_atl_in_area.atl_datetime.values[0]], falling_edge_dt])
if rising_edge_dt.size < n_section_for_current_mmsi:
    rising_edge_dt = np.concat([rising_edge_dt, [ds_atl_in_area.atl_datetime.values[-1]]])
    
event_sections = []
for isec in range(n_section_for_current_mmsi):
    pos_sec = ds_atl_in_area.sel(
        atl_datetime=slice(falling_edge_dt[isec], rising_edge_dt[isec])
    )
    event_sections.append(pos_sec)

In [ ]:
event_sections

In [ ]:
portion_event = xr.DataTree()
event_ids = np.arange(len(event_sections))
portion_event["event_id"] = event_ids
portion_event.attrs = dict(
    box_lon_min=lon_min, box_lon_max=lon_max, box_lat_min=lat_min, box_lat_max=lat_max
)

for isec, pos_sec in enumerate(event_sections):

    ds_event_i = pos_sec.rename(
        dict(atl_datetime="datetime", atl_lon="lon", atl_lat="lat")
    )
    # ds_event_i = ds_event_i.drop_vars("mmsi")
    ds_event_i["start_datetime"] = ds_event_i.datetime.min().values
    ds_event_i["stop_datetime"] = ds_event_i.datetime.max().values
    ds_event_i.attrs["event_id"] = isec

    path = f"position/event_{isec}"
    portion_event[path] = ds_event_i

In [ ]:
portion_event

In [ ]:
# Plot
fig, ax = plot_portion_tir_pos(portion_tree=portion)
ax.add_patch(
    Rectangle(
        (portion_event.box_lon_min, portion_event.box_lat_min),
        portion_event.box_lon_max - portion_event.box_lon_min,
        portion_event.box_lat_max - portion_event.box_lat_min,
        fc="none",
        ec="k",
        lw=5,
        label="Selected area",
    )
)

for name, node in portion_event.position.children.items():
    ds_position_section = node.to_dataset()
    plt.plot(
        ds_position_section.lon, ds_position_section.lat, "-x", markersize=2, label=f"Event {ds_position_section.event_id}"
    )

ax.legend()
legend = ax.get_legend()

if legend is not None:
    handles = legend.legend_handles
    labels = [text.get_text() for text in legend.get_texts()]

    legend.remove()

    ax.legend(
        handles,
        labels,
        loc="upper right",
        fontsize=16,
        ncols=2,
    )

## Chargement des signaux 

In [ ]:
ds_pressure = xr.open_dataset(ACOUPLANE_PRESSURE_NC_FPATH)
ds_pressure

## Conversion des données et calcul vecteurs temps pour slicing 

Les données stockées sont les données raw, il faut donc les convertir et fabriquer le vecteur de temps.

In [ ]:
# Build a vector of datetime for each OBS signal
obs_datetimes = build_obs_datetime_vector(ds_pressure=ds_pressure)

## Premier tri : disponibilité des données OBS 

In [ ]:
event_to_drop = []
for name, node in portion_event.position.children.items():
    ds_position_event = node.to_dataset()

    event_start = ds_position_event.start_datetime.values
    event_stop = ds_position_event.stop_datetime.values

    invalid_event = False
    for obs_id in ds_pressure.obs_id.values:
        # Check if any data is available 
        event_stops_before_recording = obs_datetimes[obs_id][0] >= event_stop
        event_starts_after_recording = obs_datetimes[obs_id][-1] <= event_start
        no_data = event_stops_before_recording or event_starts_after_recording
        if no_data:
            print(f"OBS '{obs_id}' has no data between {event_start} and {event_stop} (recording spans {obs_datetimes[obs_id][0]} -> {obs_datetimes[obs_id][-1]}).")
            invalid_event = True

    if invalid_event:
        print(f"Event {name} dropped")
        event_to_drop.append(name)

In [ ]:
n_unvalid_event = len(event_to_drop)
print(f"{n_unvalid_event}/{portion_event.event_id.size} events dropped (no data available)")

## Deuxième tri : donnée hors de la période des tirs 

<!-- NOTE : Avec un critère directement sur les dates de toutes les émissions il ne reste que 3 trajectoires disponibles et aucunes des trajectoires ne passe dans la grille. -->

<!-- Pour éviter cela on peut définir un critère moins restrictif en ne considérant que les émissions dans la zone d'intérêt. Les émissions au dessus de l'OBS 6 devraient vraisemblablement être reçues avec un faible SNR et peuvent peut être négligées. Ces émissions ont toutes lieu le 26/02 -->

In [ ]:
filter1_stop = datetime(2026, 2, 25, 16, 40, 0)
ds_pos_filtered = ds_pos.copy()
ds_pos_filtered1 = ds_pos_filtered.sel(src_datetime=slice(ds_pos_filtered.src_datetime.values[0], filter1_stop))

filter2_start = datetime(2026, 2, 26, 4, 0, 0)
ds_pos_filtered2 = ds_pos_filtered.sel(src_datetime=slice(filter2_start, ds_pos_filtered.src_datetime.values[-1]))

ds_pos_filtered1.src_datetime, ds_pos_filtered2.src_datetime

In [ ]:
valid_events = []
for name, node in portion_event.position.children.items():
    
    if name in event_to_drop:   # Dropped according to frist condition 
        continue 

    ds_position_event = node.to_dataset()

    event_start = ds_position_event.start_datetime.values
    event_stop = ds_position_event.stop_datetime.values

    # # First option : filter to avoid any emission 
    event_stops_before_first_emission = ds_pos.src_datetime.values[0] >= event_stop
    event_starts_after_last_emission = ds_pos.src_datetime.values[-1] <= event_start
    valid_event = event_stops_before_first_emission or event_starts_after_last_emission

    # # Second option keep event if emission are emitted clsoe to OBS 6 
    # event_stops_before_first_emission1 = ds_pos_filtered1.src_datetime.values[0] >= event_stop
    # event_starts_after_last_emission1 = ds_pos_filtered1.src_datetime.values[-1] <= event_start
    # valid_event1 = event_stops_before_first_emission1 or event_starts_after_last_emission1

    # event_stops_before_first_emission2 = ds_pos_filtered2.src_datetime.values[0] >= event_stop
    # event_starts_after_last_emission2 = ds_pos_filtered2.src_datetime.values[-1] <= event_start
    # valid_event2 = event_stops_before_first_emission2 or event_starts_after_last_emission2
    # valid_event = valid_event1 and valid_event2

    if valid_event:
        valid_events.append(name)

In [ ]:
valid_events

Avec la seconde option on conserve davantage d'évenements. 

In [ ]:
# Rebuild datatree with correct event names once unvalid ones have been dropped
portion_event_filter = xr.DataTree()

for e_id_new, event_name in enumerate(valid_events):
    
    path_pos = f"position/{event_name}"
    new_path_pos = f"position/event_{e_id_new}"
    old_pos_event = portion_event[path_pos].to_dataset()
    old_pos_event.attrs["event_id"] = e_id_new
    portion_event_filter[new_path_pos] = old_pos_event

    # path_press = f"pressure/{event_name}"
    # new_path_press = f"pressure/event_{e_id_new}"
    # old_press_event = portion_event[path_press].to_dataset()
    # old_press_event.attrs["event_id"] = e_id_new
    # portion_event_filter[new_path_press] = old_press_event

portion_event_filter["event_id"] = np.arange(len(valid_events))
portion_event_filter.attrs = portion_event.attrs

del portion_event
portion_event = portion_event_filter.copy()
del portion_event_filter

In [ ]:
portion_event

In [ ]:
# Plot
fig, ax = plot_portion_tir_pos(portion_tree=portion)
ax.add_patch(
    Rectangle(
        (portion_event.box_lon_min, portion_event.box_lat_min),
        portion_event.box_lon_max - portion_event.box_lon_min,
        portion_event.box_lat_max - portion_event.box_lat_min,
        fc="none",
        ec="k",
        lw=5,
        label="Selected area",
    )
)

for name, node in portion_event.position.children.items():
    ds_position_section = node.to_dataset()
    plt.plot(
        ds_position_section.lon, ds_position_section.lat, "-x", markersize=2, label=f"Event {ds_position_section.event_id}"
    )

ax.legend()
legend = ax.get_legend()

if legend is not None:
    handles = legend.legend_handles
    labels = [text.get_text() for text in legend.get_texts()]

    legend.remove()

    ax.legend(
        handles,
        labels,
        loc="upper right",
        fontsize=16,
        ncols=2,
    )

# Extraction signaux associés aux navires sélectionnés 

## Extraction et stockage 

In [ ]:
event_to_drop = []
for name, node in portion_event.position.children.items():
    ds_position_event = node.to_dataset()

    event_start = ds_position_event.start_datetime.values
    event_stop = ds_position_event.stop_datetime.values

    invalid_event = False
    for obs_id in ds_pressure.obs_id.values:
        # Check if any data is available 
        event_stops_before_recording = obs_datetimes[obs_id][0] >= event_stop
        event_starts_after_recording = obs_datetimes[obs_id][-1] <= event_start
        no_data = event_stops_before_recording or event_starts_after_recording
        if no_data:
            print(f"OBS '{obs_id}' has no data between {event_start} and {event_stop} (recording spans {obs_datetimes[obs_id][0]} -> {obs_datetimes[obs_id][-1]}).")
            invalid_event = True


    if invalid_event:
        # portion_event.drop_nodes()
        print(f"Event {name} dropped")
        event_to_drop.append(name)
        continue

    ds_pressure_event = extract_pressure_section(
        event_start,
        event_stop,
        ds_pressure,
        obs_datetimes,
    )
    ds_pressure_event = convert_raw_pressure_to_physical_units(
        ds_pressure=ds_pressure_event
    )
    
    ds_pressure_event.attrs["event_id"] = ds_position_event.event_id
    
    # Store in event portion
    path = f"pressure/event_{ds_position_event.event_id}"
    portion_event[path] = ds_pressure_event

In [ ]:
portion_event 

## Visualisation des signaux 

In [ ]:
# Plot
single_fig = False
for name, node in portion_event.pressure.children.items():
    ds_pressure_section = node.to_dataset()
    fig, axs = plot_pressure_section(ds_pressure_section=ds_pressure_section, single_fig=single_fig)
    fig.suptitle(f"Event {ds_pressure_section.event_id}")
    plt.show()

## Spectrogrames 


In [ ]:
# Plot
for name, node in portion_event.pressure.children.items():
    ds_pressure_section = node.to_dataset()
    try:
        fig, axs = plot_spectrogram_section(ds_pressure_section, nperseg=2**10, noverlap=2**9, cmap="magma")
    except:
        print(f"Data with different length event : {name}")
    fig.suptitle(f"Event {ds_pressure_section.event_id}")
    plt.show()

Il y a surement un autre navire proche OBS 7 au même instant. 

# Sauvergarde de la portion event 

In [ ]:
# Save as netcdf
fpath = f"{ACOUPLANE_PORTION_NC_ROOT_FPATH}_{portion_id}_event_atl.nc"
portion_event.attrs["portion_filepath"] = fpath
portion_event.to_netcdf(fpath)